# MGS-29 : GA MGS compose `"Default"` contre `BaseGA` mealpy — la revanche du GA

Paire 8/9 de l'[EPIC #12373](https://github.com/jsboige/CoursIA/issues/12373) (MGS vs mealpy — comparaison appariee). Suite de [MGS-22 (PSO canonique)](MGS-22-MGS-vs-Mealpy.ipynb), qui a refute l'hypothese user sur la vitesse mais revele un ecart de **qualite de convergence a budget egal** : mealpy `OriginalPSO` mediane 28,5 conflits, MGS `ParticleSwarmOptimization` mediane 43,5 — separation totale des gammes.

Ce grain specialise la comparaison au **meme algorithme conceptuel (algorithme genetique)** : compose MGS `"Default"` (celui de la colonne R1/GA de MGS-21, mediane 10,5 conflits a 8 000 evals) contre `BaseGA` de mealpy (`mealpy.evolutionary_based.GA.BaseGA`). Question directrice : *l'ecart MGS-vs-mealpy observe sur PSO est-il une propriete systematique des MGS compounds geometriques, ou un artefact du moteur PSO choisi ?*

## Plan

1. **Socle commun** : representation R1 (continu [1,10)^36), cout conflits, decode, contre-verification croisee C# <-> Python (recette heritee de MGS-22 — preuve faite dans la cellule sanity check).
2. **Moteurs** :
   - Cote MGS : compose `"Default"` (GA canonique : elitisme + crossover uniforme + mutation uniforme, le wiring par defaut de la bibliotheque).
   - Cote mealpy : `BaseGA` (l'API `mealpy.evolutionary_based.GA.BaseGA`).
3. **Bench** : 4 graines {0, 1, 7, 42} x 3 repetitions, budget ~8 000 evaluations par course (population 50, 160 generations). Medianes + min/max rapportes.
4. **Lecture** : l'ecart de qualite PSO-vs-PSO se reproduit-il GA-vs-GA ? Si oui, c'est une propriete de compound (strategie de recherche), pas du moteur PSO.

**Acceptance** : resultat « MGS perd en qualite, gagne en vitesse » ou l'inverse est un livrable valide — l'insight noyau est l'objet du grain, pas le score.


## 1. Le protocole apparie, pre-enregistre

Identique a MGS-22 pour permettre la comparaison entre paires (PSO/PSO, GA/GA, etc.).

| Element | Valeur (fixee d'avance) |
|---|---|
| **Grille** | `Easy[0]` de Sudoku_Easy51 — la meme que MGS-21 et MGS-22 : 36 cellules vides, 45 indices |
| **Representation** | R1 : vecteur continu [1,10)^36, decodage par arrondi + clamp — substrat continu identique |
| **Fonction de cout** | conflits totaux (lignes + colonnes + blocs) — la meme implementee deux fois (C# et Python) ; **la sanity check PASSE 3/3** (67/67, 71/71, 60/60 sur vecteurs temoins LCG) ; la table rapporte `conflits_C` (re-eval C# via `r.sol`, contre-mesure croisee permanente) et `conflits_P` (brut Python, audit — les deux colonnes doivent coincider) |
| **Moteurs** | GA canonique : compose `"Default"` de MetaGeneticSharp contre `BaseGA` de mealpy |
| **Budget** | ~8 000 evaluations par course : population 50 x 160 generations/epochs, evals reellement consommees instrumentees |
| **Graines** | {0, 1, 7, 42} — nommees, une par course, 4 courses par moteur |
| **Mesures** | (a) qualite : conflits finaux, mediane + min-max sur 4 graines · (b) vitesse : ms par course, mediane de 3 repetitions par graine |

**Criteres de lecture** : (1) Qualite — un moteur domine si mediane strictement inferieure ET maxima sous les minima de l'autre, sinon « comparable ». (2) Vitesse — rapport des ms/eval moyens. (3) Le verdict reprend la these directrice — chaque axe rapporte separement, aucune compensation.

In [1]:
// === MGS-29 : socle commun — DLLs MGS, grille de reference, fonction de cout ===
// Reprend le socle de MGS-22 : la representation R1 (continu + arrondi) est le substrat du bench.
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
using MetaGeneticSharp;
using GeneticSharp;
using System.Diagnostics;

// Grille facile Easy[0] de Sudoku_Easy51.txt — la MEME que MGS-21 / MGS-22.
public static string PuzzleLine29 = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";

public static int[,] ParsePuzzle29()
{
    var g = new int[9, 9];
    for (int i = 0; i < 81; i++) g[i / 9, i % 9] = PuzzleLine29[i] - '0';
    return g;
}

// Fonction de cout du bench : conflits totaux (lignes + colonnes + blocs) sur grille PLEINE.
public static int CountConflicts29(int[,] g)
{
    int conflicts = 0;
    for (int i = 0; i < 9; i++)
    {
        var row = new HashSet<int>(); var col = new HashSet<int>(); var blk = new HashSet<int>();
        for (int j = 0; j < 9; j++)
        {
            if (!row.Add(g[i, j])) conflicts++;
            if (!col.Add(g[j, i])) conflicts++;
            int br = 3 * (i / 3) + j / 3, bc = 3 * (i % 3) + j % 3;
            if (!blk.Add(g[br, bc])) conflicts++;
        }
    }
    return conflicts;
}

public static int CountEmpty29(int[,] p) { int n = 0; foreach (var v in p) if (v == 0) n++; return n; }

public static List<(int r, int c)> EmptyCells29(int[,] p)
{
    var l = new List<(int, int)>();
    for (int r = 0; r < 9; r++) for (int c = 0; c < 9; c++) if (p[r, c] == 0) l.Add((r, c));
    return l;
}

public static int[,] DecodeR1_29(double[] genes)
{
    var Puzzle = ParsePuzzle29();
    var empties = EmptyCells29(Puzzle);
    var g = (int[,])Puzzle.Clone();
    for (int k = 0; k < empties.Count; k++)
        g[empties[k].r, empties[k].c] = Math.Max(1, Math.Min(9, (int)Math.Round(genes[k])));
    return g;
}

var Puzzle29 = ParsePuzzle29();
Console.WriteLine($"Grille de reference : {CountEmpty29(Puzzle29)} cellules vides, " +
                  $"{81 - CountEmpty29(Puzzle29)} indices fixes, {EmptyCells29(Puzzle29).Count} genes R1.");


Grille de reference : 36 cellules vides, 45 indices fixes, 36 genes R1.


### Interpretation : le socle est pose

Reprend le substrat du bench MGS-22 (sans modification) :
- vecteur continu de 36 genes, range [1, 10)
- decode par arrondi + clamp vers {1, ..., 9}
- cout = conflits totaux sur la grille pleine

**Identite des couts** : cout C# et cout Python coincident — sanity check **PASSE 3/3** (67/67, 71/71, 60/60 sur vecteurs temoins LCG), contre-verif croisee de la cellule 8 IDENTIQUE. Les deux moteurs optimisent le meme objectif (conflits totaux lignes + colonnes + blocs). La double comptabilite de la cellule 9 (`conflits_C` re-eval C#, `conflits_P` brut Python) sert de contre-mesure croisee permanente : les deux colonnes coincident et toute divergence future re-signale une derive du pont. Voir cellules 10 et 11 pour le verdict rejoue sous le vrai paysage.

In [2]:
// === Moteur MGS : compound "Default" (GA canonique : EliteSelection + UniformCrossover + UniformMutation) ===
// Le compose "Default" est l'agencement GA par defaut de MetaGeneticSharp (cf colonne R1/GA de MGS-21).
public class SudokuR1Chromosome29 : ChromosomeBase
{
    private const double LO = 1.0, HI = 10.0;
    public SudokuR1Chromosome29() : base(EmptyCells29(ParsePuzzle29()).Count) { CreateGenes(); }
    public override Gene GenerateGene(int index)
        => new Gene(RandomizationProvider.Current.GetDouble(LO, HI));
    public override IChromosome CreateNew() => new SudokuR1Chromosome29();
    public double[] ToGenes() { var v = new double[Length]; for (int i = 0; i < Length; i++) v[i] = (double)GetGene(i).Value; return v; }
    public int[,] ToGrid() => DecodeR1_29(ToGenes());
}

public class SudokuR1Fitness29 : IFitness
{
    public static int Evals;
    public double Evaluate(IChromosome chromosome)
    {
        Evals++;
        return -CountConflicts29(((SudokuR1Chromosome29)chromosome).ToGrid());
    }
}

public static class Mgs29Host
{
    public static (int conflicts, int evals, double ms, double[] genes) RunGa(int seed, int popSize, int maxGens)
    {
        // Seeding AVANT creation de population : le RNG est consomme par CreateNew()
        // de chaque individu initial (lecon MGS-21 / #12071).
        FastRandomRandomization.ResetSeed(seed);
        var compound = MetaHeuristicsService.CreateMetaHeuristicByName(
            "Default", maxGens, popSize);
        var adam = new SudokuR1Chromosome29();
        var pop = new MetaPopulation(popSize, popSize, adam);
        var ga = new MetaGeneticAlgorithm(
            pop, new SudokuR1Fitness29(),
            new EliteSelection(), new UniformCrossover(0.5f), new UniformMutation(true),
            compound);
        ga.Termination = new GenerationNumberTermination(maxGens);
        SudokuR1Fitness29.Evals = 0;
        var sw = Stopwatch.StartNew();
        ga.Start();
        sw.Stop();
        var best = (SudokuR1Chromosome29)ga.BestChromosome;
        return (CountConflicts29(best.ToGrid()), SudokuR1Fitness29.Evals,
                sw.Elapsed.TotalMilliseconds, best.ToGenes());
    }
}

// Echauffement JIT (course jetee), puis course temoin graine 7.
var warmupMgs = Mgs29Host.RunGa(123, 50, 10);
var demoMgs = Mgs29Host.RunGa(7, 50, 160);
Console.WriteLine($"MGS GA Default (graine 7, temoin) : {demoMgs.Item1} conflits, " +
                  $"{demoMgs.Item2} evaluations, {demoMgs.Item3:F0} ms.");


MGS GA Default (graine 7, temoin) : 15 conflits, 6026 evaluations, 106 ms.


### Interpretation : ce que le moteur MGS expose

Trois choix de wiring portent la validite du bench :
1. **Le compteur d'evaluations instrumente** : `SudokuR1Fitness29.Evals` compte chaque appel a `Evaluate` — le budget reel est mesure, pas suppose.
2. **Le seeding avant population** : `FastRandomRandomization.ResetSeed(seed)` est appele AVANT `CreateMetaHeuristicByName`, parce que le RNG est consomme par `CreateNew()` de chaque individu initial. Un seeding apres = une autre population que la semence declaree.
3. **Le compose `Default`** : c'est l'agencement GA canonique de MetaGeneticSharp (elitisme + crossover uniforme + mutation uniforme), equivalent structurel au `BaseGA` de mealpy cote wiring.

Le seul parametre qu'on ne controle pas est la valeur concrete des operateurs de mutation (taux, distribution) — MGS utilise des valeurs par defaut, mealpy aussi, et c'est precisement ce qui rend la comparaison interessante : « out of the box, que vaut chaque bibliotheque ? ».


In [3]:
// === Le pont PythonNet : mealpy dans le meme kernel, la meme execution ===
// pythonnet 3.0.5 : la 3.1.0 cherche PyThreadState_GetUnchecked, absent de CPython 3.13 stable.
#r "nuget: pythonnet,3.0.5"
using Python.Runtime;
static string ResolvePythonDll()
{
    var env = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if (!string.IsNullOrEmpty(env) && System.IO.File.Exists(env)) return env;
    if (OperatingSystem.IsWindows())
    {
        // Installs CPython.org standards d'abord (les plus récentes portent les packages récents
        // comme mealpy), ensuite le scan LOCALAPPDATA — un Python périmé qui n'a pas mealpy
        // ne doit pas masquer une install plus récente (pb 2026-08-25 : Python310 2023 devant Python313).
        foreach (var c in new[] { @"C:\Python313\python313.dll", @"C:\Python312\python312.dll" })
            if (System.IO.File.Exists(c)) return c;
        var local = Environment.GetEnvironmentVariable("LOCALAPPDATA");
        if (!string.IsNullOrEmpty(local))
        {
            var pyDir = System.IO.Path.Combine(local, "Programs", "Python");
            if (System.IO.Directory.Exists(pyDir))
                foreach (var d in System.IO.Directory.GetDirectories(pyDir, "Python3*"))
                {
                    var hits = System.IO.Directory.GetFiles(d, "python3*.dll");
                    // Prefer python3XX.dll (versionnee, contient PyThreadState_GetUnchecked)
                    // sur python3.dll (stable ABI, qui en est depourvue en CPython 3.13).
                    var pick = "";
                    foreach (var h in hits)
                        if (System.IO.Path.GetFileName(h).Length > 11) pick = h;
                    if (pick == "" && hits.Length > 0) pick = hits[0];
                    if (pick != "") return pick;
                }
        }
        var home = Environment.GetFolderPath(Environment.SpecialFolder.UserProfile);
        foreach (var root in new[] {
                     System.IO.Path.Combine(home, "miniconda3"),
                     System.IO.Path.Combine(home, "anaconda3"),
                     @"C:\ProgramData\miniconda3",
                     @"C:\ProgramData\anaconda3" })
        {
            if (!System.IO.Directory.Exists(root)) continue;
            var hits = System.IO.Directory.GetFiles(root, "python3*.dll");
            var pick = "";
            foreach (var h in hits)
                if (System.IO.Path.GetFileName(h).Length > 11) pick = h;
            if (pick == "" && hits.Length > 0) pick = hits[0];
            if (pick != "")
            {
                var dirs = new[] { root,
                    System.IO.Path.Combine(root, "Library", "mingw-w64", "bin"),
                    System.IO.Path.Combine(root, "Library", "bin"),
                    System.IO.Path.Combine(root, "Scripts") };
                var path = Environment.GetEnvironmentVariable("PATH") ?? "";
                var toAdd = "";
                foreach (var d in dirs)
                    if (System.IO.Directory.Exists(d) && !path.Contains(d + ";"))
                        toAdd += d + ";";
                if (toAdd != "")
                    Environment.SetEnvironmentVariable("PATH", toAdd + path);
                return pick;
            }
        }
    }
    else
    {
        var libs = new[] { "/usr/lib/x86_64-linux-gnu", "/usr/lib", "/usr/local/lib", "/opt/homebrew/lib" };
        foreach (var dir in libs)
            if (System.IO.Directory.Exists(dir))
            {
                var hit = System.IO.Directory.GetFiles(dir, "libpython3.*");
                foreach (var h in hit)
                    if (h.EndsWith(".so") || h.EndsWith(".dylib")) return h;
            }
    }
    throw new System.IO.FileNotFoundException(
        "DLL Python introuvable : definir PYTHONNET_PYDLL ou installer Python 3.10+ (mealpy requis).");
}
Runtime.PythonDLL = ResolvePythonDll();
PythonEngine.Initialize();

// Le probleme Python : decode + cout reimplementes a l'identique, compteur d'evals,
// solveur mealpy BaseGA avec seed EXPLICITE en solve().
public static PyModule S29;
using (Py.GIL())
{
    S29 = Py.CreateScope();
    S29.Set("puzzle_line29", PuzzleLine29);
    S29.Exec(@"import sys
import mealpy
from mealpy.evolutionary_based.GA import BaseGA
from mealpy import Problem, FloatVar
import json as _json

puzzle = [int(ch) for ch in puzzle_line29]
empties = [(i // 9, i % 9) for i in range(81) if puzzle[i] == 0]

def decode(vec):
    g = [puzzle[r * 9:(r + 1) * 9] for r in range(9)]
    for k in range(len(empties)):
        r, c = empties[k]
        v = int(round(float(vec[k])))
        g[r][c] = max(1, min(9, v))
    return g

def cost(g):
    conflicts = 0
    for i in range(9):
        units = ([g[i][j] for j in range(9)],
                 [g[j][i] for j in range(9)],
                 [g[3 * (i // 3) + j // 3][3 * (i % 3) + j % 3] for j in range(9)])
        for unit in units:
            seen = set()
            for v in unit:
                if v in seen:
                    conflicts += 1
                seen.add(v)
    return conflicts

def cost_of_vector(vec):
    return cost(decode(vec))

PY_EVALS = [0]

class SudokuProblemGA(Problem):
    def __init__(self, bounds=None, minmax='min', **kwargs):
        super().__init__(bounds, minmax, log_to='nothing', **kwargs)
    def obj_func(self, x):
        PY_EVALS[0] += 1
        return float(cost(decode(x)))

def run_mealpy_ga(seed, pop_size, epoch):
    import time
    PY_EVALS[0] = 0
    prob = SudokuProblemGA(bounds=FloatVar(lb=(1.0,) * len(empties), ub=(10.0,) * len(empties), name='genes'), minmax='min')
    model = BaseGA(epoch=epoch, pop_size=pop_size)
    t0 = time.perf_counter()
    g_best = model.solve(prob, seed=seed)
    dt = (time.perf_counter() - t0) * 1000.0
    sol = _json.dumps([float(v) for v in g_best.solution])
    return cost(decode(g_best.solution)), PY_EVALS[0], dt, sol

def bench_mealpy(seeds_json, pop_size, epoch, reps=3):
    out = []
    for sd in _json.loads(seeds_json):
        runs = [run_mealpy_ga(sd, pop_size, epoch) for _ in range(reps)]
        cs = [r[0] for r in runs]
        es = [r[1] for r in runs]
        ts = sorted(r[2] for r in runs)
        med = ts[len(ts) // 2] if len(ts) % 2 == 1 else (ts[len(ts) // 2 - 1] + ts[len(ts) // 2]) / 2.0
        out.append({'seed': sd, 'conflicts': cs[0], 'all_same': len(set(cs)) == 1,
                    'evals': es[0], 'ms': med, 'sol': runs[0][3]})
    return _json.dumps(out)

__mealpy_ver__ = 'mealpy ' + mealpy.__version__ + ' sur Python ' + sys.version.split()[0]");
    Console.WriteLine($"Pont PythonNet actif : {S29.Get<string>("__mealpy_ver__")}");
}

// --- Sanity check : la fonction de cout est-elle la MEME des deux cotes ? ---
public static double[] LcgVector29(int seed, int n)
{
    uint state = (uint)seed;
    var v = new double[n];
    for (int i = 0; i < n; i++)
    {
        state = state * 1664525u + 1013904223u;
        v[i] = 1.0 + (state / 4294967296.0) * 9.0;
    }
    return v;
}

var witnessVectors = new[] { LcgVector29(1, 36), LcgVector29(2, 36), LcgVector29(3, 36) };
using (Py.GIL())
{
    S29.Set("__witness_json__",
        System.Text.Json.JsonSerializer.Serialize(witnessVectors.Select(v => v.ToList()).ToList()));
    S29.Exec(@"__py_costs_json__ = _json.dumps([cost_of_vector(v) for v in _json.loads(__witness_json__)])");
    var pyCosts = System.Text.Json.JsonSerializer.Deserialize<List<int>>(S29.Get<string>("__py_costs_json__"));
    bool allEqual = true;
    for (int i = 0; i < witnessVectors.Length; i++)
    {
        int csCost = CountConflicts29(DecodeR1_29(witnessVectors[i]));
        bool eq = csCost == pyCosts[i];
        allEqual &= eq;
        Console.WriteLine($"  vecteur temoin {i + 1} : cout C# = {csCost}, cout Python = {pyCosts[i]} -> {(eq ? "IDENTIQUE" : "DIFFERENT")}");
    }
    Console.WriteLine(allEqual
        ? "Sanity check PASSE : la fonction de cout est identique des deux cotes -- le bench est valide."
        : "Sanity check ECHOUE : les fonctions de cout different -- le bench serait invalide.");
}


Installing Packages pythonnet

Pont PythonNet actif : mealpy 3.0.2 sur Python 3.13.3


  vecteur temoin 1 : cout C# = 67, cout Python = 67 -> IDENTIQUE


  vecteur temoin 2 : cout C# = 71, cout Python = 71 -> IDENTIQUE


  vecteur temoin 3 : cout C# = 60, cout Python = 60 -> IDENTIQUE


Sanity check PASSE : la fonction de cout est identique des deux cotes -- le bench est valide.


### Interpretation : pourquoi la sanity check porte tout le bench

Le point fragile d'une comparaison cross-langage n'est pas le solveur — c'est la **fonction de cout**. Si le C# comptait les doublons avec une convention differente du Python, les deux moteurs n'optimiseraient pas le meme paysage, et chaque milliseconde comparee serait du bruit raffine. D'ou le protocole en deux temps :
- des vecteurs temoins deterministes (LCG ecrit a la main, 3 seeds) ;
- cout C# et cout Python calcules sur les MEMES vecteurs ;
- identite verifiee sur les 3 paires.

Si la sanity check passe, le bench est valide : ce qu'on compare, c'est bien deux optimiseurs sur le MEME paysage. Sinon, on ne compare rien — on declare forfait et on corrige le cout avant de continuer.


In [4]:
// === Moteur mealpy : course temoin + contre-verification croisee du vainqueur ===
using (Py.GIL())
{
    // Echauffement symetrique (course jetee), puis course temoin graine 7.
    S29.Exec(@"_wu_c, _wu_e, _wu_t, _wu_sol = run_mealpy_ga(123, 50, 10)
__d_c__, __d_e__, __d_t__, __d_sol__ = run_mealpy_ga(7, 50, 160)");
    int dConflicts = S29.Get<int>("__d_c__");
    int dEvals = S29.Get<int>("__d_e__");
    double dMs = S29.Get<double>("__d_t__");
    Console.WriteLine($"mealpy BaseGA (graine 7, temoin) : {dConflicts} conflits, " +
                      $"{dEvals} evaluations, {dMs:F0} ms.");

    // Contre-verification croisee : le vainqueur mealpy, decode et coste cote C#.
    var solJson = S29.Get<string>("__d_sol__");
    var genes = System.Text.Json.JsonSerializer.Deserialize<double[]>(solJson);
    int csRecheck = CountConflicts29(DecodeR1_29(genes));
    Console.WriteLine($"Contre-verif croisee : cout C# du meilleur mealpy = {csRecheck} " +
                      $"(Python rapporte {dConflicts}) -> {(csRecheck == dConflicts ? "IDENTIQUE" : "DIFFERENT")}");
}


mealpy BaseGA (graine 7, temoin) : 16 conflits, 8050 evaluations, 1377 ms.


Contre-verif croisee : cout C# du meilleur mealpy = 16 (Python rapporte 16) -> IDENTIQUE


In [5]:
// === LE BENCH : 2 moteurs x 4 graines {0,1,7,42}, population 50, 160 generations ===
public class BenchRow29
{
    public int seed { get; set; }
    public int conflicts { get; set; }
    public bool all_same { get; set; }
    public int evals { get; set; }
    public double ms { get; set; }
    public string sol { get; set; }
}

int[] Seeds29 = { 0, 1, 7, 42 };

// --- Cote MGS (C#) : 3 repetitions par graine, ms = mediane ---
var mgsRows = new List<(int seed, int conflicts, int evals, double ms, bool allSame)>();
foreach (var sd in Seeds29)
{
    var runs3 = new List<(int c, int e, double t)>();
    for (int rep = 0; rep < 3; rep++)
    {
        var r = Mgs29Host.RunGa(sd, 50, 160);
        runs3.Add((r.Item1, r.Item2, r.Item3));
    }
    var times = runs3.Select(x => x.t).OrderBy(t => t).ToList();
    double med = times[1];
    mgsRows.Add((sd, runs3[0].c, runs3[0].e, med, runs3.All(x => x.c == runs3[0].c)));
}

// --- Cote mealpy (Python, boucle unique dans le scope) ---
string mealpyJson;
using (Py.GIL())
{
    S29.Set("__seeds_json__", System.Text.Json.JsonSerializer.Serialize(Seeds29.ToList()));
    S29.Exec(@"__bench_json__ = bench_mealpy(__seeds_json__, 50, 160)");
    mealpyJson = S29.Get<string>("__bench_json__");
}
var mealpyRows = System.Text.Json.JsonSerializer.Deserialize<List<BenchRow29>>(mealpyJson);

// --- Table ---
static double Median29(List<int> xs)
{
    var s = xs.OrderBy(x => x).ToList();
    return (s.Count % 2 == 1) ? s[s.Count / 2] : (s[s.Count / 2 - 1] + s[s.Count / 2]) / 2.0;
}

// --- Cote mealpy : double comptabilite du cout (re-eval C# + brut Python) via r.sol ---
// Contre-mesure croisee permanente : chaque r.sol est re-evalue en C# via
// CountConflicts29(DecodeR1_29) ; conflits_C doit egaler conflits_P (audit), toute
// divergence future re-signale une derive du pont.
var mealpyReeval = new List<(int seed, int conflictsCs, int conflictsPy, int evals, double ms, bool allSame)>();
foreach (var r in mealpyRows)
{
    var genesPy = System.Text.Json.JsonSerializer.Deserialize<double[]>(r.sol);
    int csCost = CountConflicts29(DecodeR1_29(genesPy));
    mealpyReeval.Add((r.seed, csCost, r.conflicts, r.evals, r.ms, r.all_same));
}

Console.WriteLine($"{"moteur",-9} {"graine",6} {"conflits_C",9} {"conflits_P",10} {"evals",7} {"ms",8} {"ms/eval",8}");
foreach (var r in mgsRows)
    Console.WriteLine($"{"MGS",-9} {r.seed,6} {r.conflicts,9} {"-",10} {r.evals,7} {r.ms,8:F0} {r.ms / r.evals,8:F3}");
foreach (var r in mealpyReeval)
    Console.WriteLine($"{"mealpy",-9} {r.seed,6} {r.conflictsCs,9} {r.conflictsPy,10} {r.evals,7} {r.ms,8:F0} {r.ms / r.evals,8:F3}");

var mgsC = mgsRows.Select(r => r.conflicts).ToList();
var mpC = mealpyReeval.Select(r => r.conflictsCs).ToList();
double mgsMsEval = mgsRows.Average(r => r.ms / r.evals);
double mpMsEval = mealpyRows.Average(r => r.ms / r.evals);
Console.WriteLine();
Console.WriteLine($"MGS    : mediane conflits {Median29(mgsC):F1} (min {mgsC.Min()}, max {mgsC.Max()}), ms/eval moyen {mgsMsEval:F3}");
Console.WriteLine($"mealpy : mediane conflits {Median29(mpC):F1} (min {mpC.Min()}, max {mpC.Max()}), ms/eval moyen {mpMsEval:F3}");
Console.WriteLine($"Rapport ms/eval mealpy/MGS : {mpMsEval / mgsMsEval:F2}x");
int detMgs = mgsRows.Count(r => r.allSame) + mealpyRows.Count(r => r.all_same);
Console.WriteLine($"Determinisme : conflits identiques sur les 3 repetitions pour {detMgs}/8 paires graine-moteur.");


moteur    graine conflits_C conflits_P   evals       ms  ms/eval


MGS            0        12          -    6066      110    0,018


MGS            1        18          -    5942       84    0,014


MGS            7        15          -    6026      111    0,018


MGS           42        12          -    6082      113    0,019


...

mealpy        42        15         15    8050      953    0,118


MGS    : mediane conflits 13,5 (min 12, max 18), ms/eval moyen 0,017


mealpy : mediane conflits 12,5 (min 9, max 16), ms/eval moyen 0,136


Rapport ms/eval mealpy/MGS : 7,83x


Determinisme : conflits identiques sur les 3 repetitions pour 8/8 paires graine-moteur.


### Lecture du croisement GA-vs-GA

Les mesures ci-dessus repondent a la these directrice de l'EPIC en miroir de MGS-22 :
- Si MGS `"Default"` (compose GA) montre le meme type d'ecart que `ParticleSwarmOptimization` face a mealpy (qualite inferieure a budget egal, ex-aequo ou avantage en vitesse), c'est une **propriete systematique des MGS compounds** : la bibliotheque minimise plus lentement le cout que les bibliotheques de reference Python.
- Si au contraire l'ecart se referme (qualite comparable ou inverse), c'est un **artefact du moteur PSO** : le compose geometrique de particule a une specificite que le GA pur n'a pas.

Dans les deux cas, l'insight est « systematique vs specifique » — c'est ce qu'aucune paire isolee ne peut trancher. Cette mesure est la deuxieme d'un futur tableau 5x4 (5 paires x 4 graines minimum).

**Lecture des chiffres (verdict rejoue sous le vrai paysage)** : **qualite comparable** — mealpy 12,5 [9-16] vs MGS 13,5 [12-18], medianes proches et etendues largement chevauchees (le critere pre-enregistre exige mediane strictement inferieure ET maxima sous les minima : non verifie, mealpy garde le max 16 au-dessus du min MGS 12). **Vitesse : MGS devant**, 7,83x moins cher par evaluation (0,017 vs 0,136 ms/eval). Determinisme 8/8. Pour la these directrice : l'ecart de qualite du PSO (separation totale en MGS-22) **ne se reproduit pas** GA contre GA — c'etait un artefact du moteur PSO, pas une propriete systematique des MGS compounds ; le GA compose tient la comparaison qualite tout en gardant l'avantage vitesse.


## Resume et suite pour l'EPIC

**Resultat experimental** : voir la table de la cellule 9. La comparaison se lit sur deux axes :
1. Qualite (conflits finaux, mediane + min-max sur 4 graines) : **comparable** — mealpy 12,5 [9-16] vs MGS 13,5 [12-18], etendues chevauchees
2. Vitesse (ms/eval moyen) : **MGS devant**, 7,83x moins cher par evaluation

**Place dans l'Epic** : MGS-29 est la **paire 8/9** (Default GA compose MGS vs BaseGA mealpy) — voir [EPIC #12373](https://github.com/jsboige/CoursIA/issues/12373). Paires deja livrees (MGS-22 a MGS-28) : PSO, DifferentialEvolution, SimulatedAnnealing, WhaleOptimisation, EquilibriumOptimizer, ForensicBasedInvestigation, BareBonesPSO. Restant apres ce grain : **paire 9 (ScatterSearch)**.

**Contre-mesure croisee** : `conflits_C` / `conflits_P` sont double-comptabilisees et coincident sur les 4 graines ; toute divergence re-signale une derive du pont.

**Suite logique** : ScatterSearch est un compound qui n'a pas de jumeau evident dans mealpy — le grain commencera par etablir s'il en existe un, et conclura honnetement s'il n'y en a pas (cf ligne 7 de l'EPIC).